In [5]:
# !pip install --upgrade pip

# # Torch estable
# !pip install torch==2.1.1 --index-url https://download.pytorch.org/whl/cu121

# # Ecosistema HF estable
# !pip install --upgrade transformers
# !pip install --upgrade tokenizers
# !pip install --upgrade datasets
# !pip install --upgrade huggingface_hub
# !pip install --upgrade accelerate
# !pip install --upgrade bitsandbytes
# !pip install --upgrade peft

# # Dependencias extra
# !pip install --upgrade fsspec
# !pip install --upgrade pyarrow
# !pip install --upgrade lxml
# !pip install --upgrade sacrebleu

# !pip install -i https://test.pypi.org/simple/ lowresource-llm-evaluation==0.2.6

In [1]:
import os
import bitsandbytes
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model
from lowresource_llm_evaluation import benchmark
from lowresource_llm_evaluation.interferenciaLinguistica import loadLexicon
import pandas as pd
import numpy as np
import torch
from huggingface_hub import login
import time
import json
import gc
from dotenv import load_dotenv

base = "./"
load_dotenv(base + "secrets.env")
login()#token=os.getenv("HF_TOKEN"))

2026-04-05 21:13:00.129713: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-05 21:13:00.129886: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-05 21:13:00.268355: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-05 21:13:00.554481: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-05 21:13:02.707279: W tensorflow/compiler/tf2

In [ ]:
def clean_graphics_card():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    gc.collect()
    torch.cuda.empty_cache()

def load_gallego():
    with open(base + "EvalDatasets/Raw/idioms_train_es.txt", "r", encoding="utf-8") as fEsp:
        esp = fEsp.readlines()

    with open(base + "EvalDatasets/Raw/idioms_train_gl.txt", "r", encoding="utf-8") as fGl:
        gl = fGl.readlines()

    with open(base + "EvalDatasets/Raw/idioms_test_es.txt", "r", encoding="utf-8") as fEsp:
        espTest = fEsp.readlines()

    with open(base + "EvalDatasets/Raw/idioms_test_gl.txt", "r", encoding="utf-8") as fGl:
        glTest = fGl.readlines()
    return pd.DataFrame(np.array((esp + espTest, gl + glTest)).T, columns=["es","gl"])

def evaluate_benchmark(model_name, idioma, token, N= 20, device="cuda", debug=False, remote_code=True):
    
    # 1. Define the 4-bit quantization configuration
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=  torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    # 2. Load the tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=remote_code)

    # 3. Load the pre-trained language model with quantization
    model = AutoModelForCausalLM.from_pretrained(
                model_name,
                quantization_config=bnb_config,
                trust_remote_code=remote_code,
                tie_word_embeddings=False, # Added to silence the warning about tied weights
                token = token,
                device_map="auto"
            )
    
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id


    print(f"Tokenizer loaded: {tokenizer.__class__.__name__}")
    print(f"Model loaded with 4-bit quantization: {model.__class__.__name__}")
    print(f"Model device: {model.device}")
    codigos = {"aranes": "aran" , 
               "asturiano": "ast", 
               "gallego": "gl"}
    df_textos = {"aranes": pd.read_parquet("hf://datasets/projecte-aina/ES-OC_Parallel_Corpus/es-arn_corpus.parquet").head(N) , 
            "asturiano": pd.read_parquet("hf://datasets/projecte-aina/ES-AST_Parallel_Corpus/es-ast_corpus.parquet").head(N), 
            "gallego": load_gallego().head(N) }
    results = benchmark(model, tokenizer, 
            df_textos = df_textos[idioma],
            lang_eval= codigos[idioma],
            df_huecos=  pd.read_csv(base + f"EvalDatasets/Huecos/{idioma}.csv").head(N),
            df_anotado = pd.read_csv(base + f"EvalDatasets/Anotado/{idioma}.csv").head(N),
            lexicon_target = loadLexicon(base + f"lexicons/{codigos[idioma]}.txt"),
            lexicons_comparison = {"es": loadLexicon(base + f"lexicons/es.txt"), "fr": loadLexicon(base + f"lexicons/fr.txt")},
            roundtrip_langs= ["es"],
            debug=debug)
    # Lberamos GPU
    try:
        model.to("cpu")
        del model
        del tokenizer
        clean_graphics_card()
    except Exception as e:
        print("Borrar el modelo ha fallado")
        print(e)
    return results

In [ ]:
import torch
print(torch.cuda.is_available())
import transformers
print(transformers.__version__)

True
4.40.2


# Aranés

## Mistral 7B 

In [4]:
idioma = "aranes"
modelo = "mistralai/Mistral-7B-Instruct-v0.3"
resultados = evaluate_benchmark(modelo, idioma, os.getenv("HF_TOKEN"), 100)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers


...

TRADUCCIÓN ROUND TRIP acabado  en 22.35
Empezando VOCABULARIO


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

VOCABULARIO acabado  en 7.12
Empezando ORTOGRAFÍA


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

ORTOGRAFÍA acabado en 13.18

                        Evaluación de Calidad de Lengua                         

+-----------------+------------------------------------------------------+
| Clave           | Valor                                                |
+-----------------+------------------------------------------------------+
| ttr             | 0.4468280592632685                                   |
| entropy         | 6.797889369623699                                    |
| ngram_overlap   | 0.0003952569169960474                                |
| freq_target     | 0.38439681172295503                                  |
| freq_comparison | {'fr': 0.3006472834224407, 'es': 0.4206148443450034} |
| calidad         | 0.14916084525161938                                  |
+-----------------+------------------------------------------------------+

                            Evaluación de Traducción                            

+------+--------------------+
| Clave | Valor           

## Salamandra

In [5]:
idioma = "aranes"
modelo = "BSC-LT/salamandra-7b-instruct"
resultados = evaluate_benchmark(modelo, idioma, os.getenv("HF_TOKEN"), 100)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.81M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/19.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/513 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/730 [00:00<?, ?B/s]

...

TRADUCCIÓN ROUND TRIP acabado  en 5.31
Empezando VOCABULARIO


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

VOCABULARIO acabado  en 1.23
Empezando ORTOGRAFÍA


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

ORTOGRAFÍA acabado en 4.34

                        Evaluación de Calidad de Lengua                         

+-----------------+------------------------------------------------------+
| Clave           | Valor                                                |
+-----------------+------------------------------------------------------+
| ttr             | 0.7562279887831306                                   |
| entropy         | 7.279569591794859                                    |
| ngram_overlap   | 0.0                                                  |
| freq_target     | 0.35709921241041154                                  |
| freq_comparison | {'fr': 0.2654210737670581, 'es': 0.5132812771654367} |
| calidad         | 0.2279249210510482                                   |
+-----------------+------------------------------------------------------+

                            Evaluación de Traducción                            

+------+--------------------+
| Clave | Valor            

## Gemma

In [6]:
idioma = "aranes"
modelo = "google/gemma-7b-it"
resultados = evaluate_benchmark(modelo, idioma, os.getenv("HF_TOKEN"), 100)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

You are calling `save_pretrained` to a 4-bit converted model, but your `bitsandbytes` version doesn't support it. If you want to save 4-bit models, make sure to have `bitsandbytes>=0.41.3` installed.


Tokenizer loaded: GemmaTokenizerFast
Model loaded with 4-bit quantization: GemmaForCausalLM
Model device: cuda:0
Empezando CALIDAD DE LENGUA
CALIDAD DE LENGUA acabada en 5.69 minutos
Empezando TRADUCCIÓN DIRECTA
TRADUCCIÓN DIRECTA acabada en 14.94
Empezando TRADUCCIÓN ROUND TRIP
TRADUCCIÓN ROUND TRIP acabado  en 30.63
Empezando VOCABULARIO
VOCABULARIO acabado  en 7.93
Empezando ORTOGRAFÍA


You shouldn't move a model when it is dispatched on multiple devices.


ORTOGRAFÍA acabado en 15.14

                        Evaluación de Calidad de Lengua                         

+-----------------+-----------------------------------------------------------+
| Clave           | Valor                                                     |
+-----------------+-----------------------------------------------------------+
| ttr             | 0.051090110874326224                                      |
| entropy         | 1.5897732835934533                                        |
| ngram_overlap   | 0.0                                                       |
| freq_target     | 0.7499187409853281                                        |
| freq_comparison | {'fr': 0.001548099892299094, 'es': 0.0015812282545093162} |
| calidad         | 0.0016753372911679828                                     |
+-----------------+-----------------------------------------------------------+

                            Evaluación de Traducción                            

+-----

## Qwen

In [7]:
idioma = "aranes"
modelo = "Qwen/Qwen2.5-7B-Instruct"
resultados = evaluate_benchmark(modelo, idioma, os.getenv("HF_TOKEN"), 100)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


...

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

You are calling `save_pretrained` to a 4-bit converted model, but your `bitsandbytes` version doesn't support it. If you want to save 4-bit models, make sure to have `bitsandbytes>=0.41.3` installed.


Tokenizer loaded: Qwen2TokenizerFast
Model loaded with 4-bit quantization: Qwen2ForCausalLM
Model device: cuda:0
Empezando CALIDAD DE LENGUA
CALIDAD DE LENGUA acabada en 3.9 minutos
Empezando TRADUCCIÓN DIRECTA
TRADUCCIÓN DIRECTA acabada en 5.64
Empezando TRADUCCIÓN ROUND TRIP
TRADUCCIÓN ROUND TRIP acabado  en 21.05
Empezando VOCABULARIO
VOCABULARIO acabado  en 7.24
Empezando ORTOGRAFÍA


You shouldn't move a model when it is dispatched on multiple devices.


ORTOGRAFÍA acabado en 14.21

                        Evaluación de Calidad de Lengua                         

+-----------------+-----------------------------------------------------+
| Clave           | Valor                                               |
+-----------------+-----------------------------------------------------+
| ttr             | 0.41820271379298085                                 |
| entropy         | 6.5815273711871685                                  |
| ngram_overlap   | 0.0005970149253731343                               |
| freq_target     | 0.504569996195515                                   |
| freq_comparison | {'fr': 0.352112300948153, 'es': 0.5451110272437617} |
| calidad         | 0.1498213344443171                                  |
+-----------------+-----------------------------------------------------+

                            Evaluación de Traducción                            

+------+--------------------+
| Clave | Valor              |
+----

# Asturiano

## Mistral 7B 

In [8]:
idioma = "asturiano"
modelo = "mistralai/Mistral-7B-Instruct-v0.3"
resultados = evaluate_benchmark(modelo, idioma, os.getenv("HF_TOKEN"), 100)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers
You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama.LlamaTokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


...

TRADUCCIÓN ROUND TRIP acabado  en 21.98
Empezando VOCABULARIO


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

VOCABULARIO acabado  en 7.29
Empezando ORTOGRAFÍA


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

ORTOGRAFÍA acabado en 11.21

                        Evaluación de Calidad de Lengua                         

+-----------------+-------------------------------------------------------+
| Clave           | Valor                                                 |
+-----------------+-------------------------------------------------------+
| ttr             | 0.4914321467893462                                    |
| entropy         | 6.913235337452818                                     |
| ngram_overlap   | 0.00045454545454545455                                |
| freq_target     | 0.5995110740458992                                    |
| freq_comparison | {'es': 0.8804156416156468, 'fr': 0.37209721866459444} |
| calidad         | 0.17577001291430808                                   |
+-----------------+-------------------------------------------------------+

                            Evaluación de Traducción                            

+------+--------------------+
| Clave | Valor 

## Salamandra

In [9]:
idioma = "asturiano"
modelo = "BSC-LT/salamandra-7b-instruct"
resultados = evaluate_benchmark(modelo, idioma, os.getenv("HF_TOKEN"), 100)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.81M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/19.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/513 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/730 [00:00<?, ?B/s]

...

TRADUCCIÓN ROUND TRIP acabado  en 25.31
Empezando VOCABULARIO


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

VOCABULARIO acabado  en 3.94
Empezando ORTOGRAFÍA


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

ORTOGRAFÍA acabado en 13.81

                        Evaluación de Calidad de Lengua                         

+-----------------+-------------------------------------------------------+
| Clave           | Valor                                                 |
+-----------------+-------------------------------------------------------+
| ttr             | 0.7452660886194971                                    |
| entropy         | 8.1967495626084                                       |
| ngram_overlap   | 0.0016217697848299809                                 |
| freq_target     | 0.451596747231602                                     |
| freq_comparison | {'es': 0.6654928993557442, 'fr': 0.19709178014684256} |
| calidad         | 0.17777349362498818                                   |
+-----------------+-------------------------------------------------------+

                            Evaluación de Traducción                            

+------+--------------------+
| Clave | Valor 

## Gemma

In [8]:
idioma = "asturiano"
modelo = "google/gemma-7b-it"
resultados = evaluate_benchmark(modelo, idioma, os.getenv("HF_TOKEN"), 100)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of GemmaForCausalLM were not initialized from the model checkpoint at google/gemma-7b-it and are newly initialized: ['lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
You are calling `save_pretrained` to a 4-bit converted model, but your `bitsandbytes` version doesn't support it. If you want to save 4-bit models, make sure to have `bitsandbytes>=0.41.3` installed.


Tokenizer loaded: GemmaTokenizerFast
Model loaded with 4-bit quantization: GemmaForCausalLM
Model device: cuda:0
Empezando CALIDAD DE LENGUA
CALIDAD DE LENGUA acabada en 5.67 minutos
Empezando TRADUCCIÓN DIRECTA
TRADUCCIÓN DIRECTA acabada en 15.13
Empezando TRADUCCIÓN ROUND TRIP
TRADUCCIÓN ROUND TRIP acabado  en 30.7
Empezando VOCABULARIO
VOCABULARIO acabado  en 7.9
Empezando ORTOGRAFÍA


You shouldn't move a model when it is dispatched on multiple devices.


ORTOGRAFÍA acabado en 15.18

                        Evaluación de Calidad de Lengua                         

+-----------------+---------------------------------------------------------+
| Clave           | Valor                                                   |
+-----------------+---------------------------------------------------------+
| ttr             | 0.5071956325806686                                      |
| entropy         | 6.784265422479265                                       |
| ngram_overlap   | 0.0                                                     |
| freq_target     | 0.042188620366409045                                    |
| freq_comparison | {'fr': 0.007158821572763886, 'es': 0.05117134859101972} |
| calidad         | 0.16042523425389738                                     |
+-----------------+---------------------------------------------------------+

                            Evaluación de Traducción                            

+------+------------------

## Qwen

In [9]:
idioma = "asturiano"
modelo = "Qwen/Qwen2.5-7B-Instruct"
resultados = evaluate_benchmark(modelo, idioma, os.getenv("HF_TOKEN"), 100)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

You are calling `save_pretrained` to a 4-bit converted model, but your `bitsandbytes` version doesn't support it. If you want to save 4-bit models, make sure to have `bitsandbytes>=0.41.3` installed.


Tokenizer loaded: Qwen2TokenizerFast
Model loaded with 4-bit quantization: Qwen2ForCausalLM
Model device: cuda:0
Empezando CALIDAD DE LENGUA
CALIDAD DE LENGUA acabada en 3.69 minutos
Empezando TRADUCCIÓN DIRECTA
TRADUCCIÓN DIRECTA acabada en 9.19
Empezando TRADUCCIÓN ROUND TRIP
TRADUCCIÓN ROUND TRIP acabado  en 17.36
Empezando VOCABULARIO
VOCABULARIO acabado  en 5.62
Empezando ORTOGRAFÍA


You shouldn't move a model when it is dispatched on multiple devices.


ORTOGRAFÍA acabado en 9.29

                        Evaluación de Calidad de Lengua                         

+-----------------+------------------------------------------------------+
| Clave           | Valor                                                |
+-----------------+------------------------------------------------------+
| ttr             | 0.3361233708500351                                   |
| entropy         | 6.196606173473731                                    |
| ngram_overlap   | 0.0014388489208633094                                |
| freq_target     | 0.5661339982587829                                   |
| freq_comparison | {'fr': 0.3803529866838442, 'es': 0.8164413907479684} |
| calidad         | 0.10342948528302245                                  |
+-----------------+------------------------------------------------------+

                            Evaluación de Traducción                            

+------+-------------------+
| Clave | Valor             

# Gallego

## Mistral

In [4]:
idioma = "gallego"
modelo = "mistralai/Mistral-7B-Instruct-v0.3"
resultados = evaluate_benchmark(modelo, idioma, os.getenv("HF_TOKEN"), 100)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers


...

TRADUCCIÓN ROUND TRIP acabado  en 20.31
Empezando VOCABULARIO


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

VOCABULARIO acabado  en 6.01
Empezando ORTOGRAFÍA


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

ORTOGRAFÍA acabado en 14.2

                        Evaluación de Calidad de Lengua                         

+-----------------+-----------------------------------------------------+
| Clave           | Valor                                               |
+-----------------+-----------------------------------------------------+
| ttr             | 0.4723147557811627                                  |
| entropy         | 6.6346865137731825                                  |
| ngram_overlap   | 0.0                                                 |
| freq_target     | 0.7690293619570452                                  |
| freq_comparison | {'fr': 0.3428429691808419, 'es': 0.660768495605127} |
| calidad         | 0.260754335282798                                   |
+-----------------+-----------------------------------------------------+

                            Evaluación de Traducción                            

+------+--------------------+
| Clave | Valor              |
+-----

## Salamandra

In [5]:
idioma = "gallego"
modelo = "BSC-LT/salamandra-7b-instruct"
resultados = evaluate_benchmark(modelo, idioma, os.getenv("HF_TOKEN"), 100)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.81M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/19.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/513 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/730 [00:00<?, ?B/s]

...

TRADUCCIÓN ROUND TRIP acabado  en 29.23
Empezando VOCABULARIO


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

VOCABULARIO acabado  en 9.09
Empezando ORTOGRAFÍA


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

ORTOGRAFÍA acabado en 14.48

                        Evaluación de Calidad de Lengua                         

+-----------------+------------------------------------------------------+
| Clave           | Valor                                                |
+-----------------+------------------------------------------------------+
| ttr             | 0.6443754433996991                                   |
| entropy         | 7.664332105244661                                    |
| ngram_overlap   | 0.0026302959817206765                                |
| freq_target     | 0.787255014413072                                    |
| freq_comparison | {'fr': 0.2603835045113977, 'es': 0.5272114627175764} |
| calidad         | 0.3347742148779034                                   |
+-----------------+------------------------------------------------------+

                            Evaluación de Traducción                            

+------+--------------------+
| Clave | Valor           

## Gemma

In [6]:
idioma = "gallego"
modelo = "google/gemma-7b-it"
resultados = evaluate_benchmark(modelo, idioma, os.getenv("HF_TOKEN"), 100)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

model-00001-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/2.11G [00:00<?, ?B/s]

`config.hidden_act` is ignored, you should use `config.hidden_activation` instead.
Gemma's activation function will be set to `gelu_pytorch_tanh`. Please, use
`config.hidden_activation` if you want to override this behaviour.
See https://github.com/huggingface/transformers/pull/29402 for more details.


...

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

You are calling `save_pretrained` to a 4-bit converted model, but your `bitsandbytes` version doesn't support it. If you want to save 4-bit models, make sure to have `bitsandbytes>=0.41.3` installed.


Tokenizer loaded: GemmaTokenizerFast
Model loaded with 4-bit quantization: GemmaForCausalLM
Model device: cuda:0
Empezando CALIDAD DE LENGUA
CALIDAD DE LENGUA acabada en 5.64 minutos
Empezando TRADUCCIÓN DIRECTA
TRADUCCIÓN DIRECTA acabada en 14.79
Empezando TRADUCCIÓN ROUND TRIP
TRADUCCIÓN ROUND TRIP acabado  en 29.82
Empezando VOCABULARIO
VOCABULARIO acabado  en 7.69
Empezando ORTOGRAFÍA


You shouldn't move a model that is dispatched using accelerate hooks.


ORTOGRAFÍA acabado en 14.69

                        Evaluación de Calidad de Lengua                         

+-----------------+--------------------------------------------------------+
| Clave           | Valor                                                  |
+-----------------+--------------------------------------------------------+
| ttr             | 0.25542355139812095                                    |
| entropy         | 4.232991929982486                                      |
| ngram_overlap   | 0.0                                                    |
| freq_target     | 0.007829863470282632                                   |
| freq_comparison | {'es': 0.38843386871540986, 'fr': 0.39445415466501066} |
| calidad         | 0.016310305859302737                                   |
+-----------------+--------------------------------------------------------+

                            Evaluación de Traducción                            

+------+---------------------+
| Cla

## Qwen

In [7]:
idioma = "gallego"
modelo = "Qwen/Qwen2.5-7B-Instruct"
resultados = evaluate_benchmark(modelo, idioma, os.getenv("HF_TOKEN"), 100)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


...

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

You are calling `save_pretrained` to a 4-bit converted model, but your `bitsandbytes` version doesn't support it. If you want to save 4-bit models, make sure to have `bitsandbytes>=0.41.3` installed.


Tokenizer loaded: Qwen2TokenizerFast
Model loaded with 4-bit quantization: Qwen2ForCausalLM
Model device: cuda:0
Empezando CALIDAD DE LENGUA
CALIDAD DE LENGUA acabada en 4.75 minutos
Empezando TRADUCCIÓN DIRECTA
TRADUCCIÓN DIRECTA acabada en 6.77
Empezando TRADUCCIÓN ROUND TRIP
TRADUCCIÓN ROUND TRIP acabado  en 15.2
Empezando VOCABULARIO
VOCABULARIO acabado  en 4.32
Empezando ORTOGRAFÍA


You shouldn't move a model that is dispatched using accelerate hooks.


ORTOGRAFÍA acabado en 5.91

                        Evaluación de Calidad de Lengua                         

+-----------------+------------------------------------------------------+
| Clave           | Valor                                                |
+-----------------+------------------------------------------------------+
| ttr             | 0.4223599313485179                                   |
| entropy         | 6.796059536629125                                    |
| ngram_overlap   | 0.00071301247771836                                  |
| freq_target     | 0.8904331183520557                                   |
| freq_comparison | {'es': 0.6889028112535103, 'fr': 0.3150030968896901} |
| calidad         | 0.2549139121632192                                   |
+-----------------+------------------------------------------------------+

                            Evaluación de Traducción                            

+------+--------------------+
| Clave | Valor            